# 04 — Relatório Final e Análise dos Resultados

Consolida os resultados de `results/elasticsearch/` em tabelas (CSV, Markdown e LaTeX), gráficos para o texto
do TCC, teste das hipóteses e recomendações. Este notebook substitui o antigo `scripts/05_generate_report.py`
e a análise final anterior.

**Pré-requisito:** `results/elasticsearch/all_results.json` (notebook 03).

Este notebook não usa Elasticsearch nem GPU; a variável `DEVICE` não tem efeito aqui.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# ── Configuração única ───────────────────────────────────────────────
DEVICE = "auto"                     # "cpu" | "cuda" | "auto"
DRIVE_PROJECT_DIR = "MyDrive/tcc"   # pasta do projeto no Google Drive (só Colab)
# ─────────────────────────────────────────────────────────────────────

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive") / DRIVE_PROJECT_DIR
    %pip install -q polars pyarrow matplotlib
else:
    PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / "src").exists(), f"Pasta 'src' não encontrada em {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))

from src.device import build_runtime_config, collect

RUNTIME = build_runtime_config(DEVICE)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Colab   : {IN_COLAB}")
print(f"Device  : {RUNTIME.device} (batch_size={RUNTIME.batch_size}, max_seq_length={RUNTIME.max_seq_length})")

In [ ]:
import json

import matplotlib.pyplot as plt
import polars as pl

from src.models.registry import list_models

RESULTS_DIR = PROJECT_ROOT / "results" / "elasticsearch"
REPORT_DIR = PROJECT_ROOT / "results" / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = [5, 10, 20, 45]
LATENCY_BUDGET_MS = 50      # limite usado no cenário de recomendação 2

MODEL_ORDER = ["bm25"] + [cfg.alias for cfg in list_models()]
DISPLAY = {"bm25": "BM25"} | {cfg.alias: cfg.display_name for cfg in list_models()}
palette = plt.get_cmap("tab10")
MODEL_COLORS = {model: palette(i) for i, model in enumerate(MODEL_ORDER)}
STRATEGY_STYLES = {
    "bm25": {"linestyle": "--", "marker": "s"},
    "knn": {"linestyle": "-", "marker": "o"},
    "hybrid": {"linestyle": "-.", "marker": "^"},
}

plt.rcParams.update({"figure.dpi": 150, "font.size": 11, "axes.titlesize": 13, "legend.fontsize": 9})

## 1. Carregamento dos resultados

In [ ]:
results_path = RESULTS_DIR / "all_results.json"
assert results_path.exists(), f"{results_path} não encontrado. Execute o notebook 03 primeiro."
all_results = json.loads(results_path.read_text(encoding="utf-8"))
assert all_results, "Nenhum resultado disponível."

df = (
    pl.DataFrame(
        [
            {
                "model": r["model"],
                "strategy": r["strategy"],
                "n_queries": r.get("n_queries"),
                "latency_ms_mean": r.get("latency_ms_mean"),
                "latency_ms_p95": r.get("latency_ms_p95"),
                **{f"ndcg@{k}": r["metrics"].get(f"ndcg@{k}") for k in K_VALUES},
            }
            for r in all_results
        ]
    )
    .with_columns(_order=pl.col("model").replace_strict({m: i for i, m in enumerate(MODEL_ORDER)}, default=99))
    .sort("_order", "strategy")
    .drop("_order")
)
print(f"Experimentos carregados: {df.height}")
df

## 2. Tabela comparativa final

Exportada em CSV, Markdown e LaTeX. No LaTeX, o melhor valor de cada coluna nDCG fica em negrito.

In [ ]:
def fmt(value: float | None, digits: int = 4) -> str:
    return "—" if value is None else f"{value:.{digits}f}"


def latex_escape(text: str) -> str:
    return text.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")


headers = ["Modelo", "Estratégia", *[f"nDCG@{k}" for k in K_VALUES], "Latência (ms)"]
best_per_k = {k: df[f"ndcg@{k}"].max() for k in K_VALUES}

md_rows, tex_rows = [], []
for row in df.iter_rows(named=True):
    name = DISPLAY.get(row["model"], row["model"])
    cells_md = [name, row["strategy"], *[fmt(row[f"ndcg@{k}"]) for k in K_VALUES], fmt(row["latency_ms_mean"], 1)]
    cells_tex = [latex_escape(name), row["strategy"]]
    for k in K_VALUES:
        text = fmt(row[f"ndcg@{k}"])
        cells_tex.append(rf"\textbf{{{text}}}" if row[f"ndcg@{k}"] == best_per_k[k] else text)
    cells_tex.append(fmt(row["latency_ms_mean"], 1))
    md_rows.append(cells_md)
    tex_rows.append(cells_tex)

markdown_table = "\n".join(
    [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
        *("| " + " | ".join(r) + " |" for r in md_rows),
    ]
)
latex_table = "\n".join(
    [
        r"\begin{tabular}{ll" + "r" * (len(headers) - 2) + "}",
        r"\toprule",
        " & ".join(headers) + r" \\",
        r"\midrule",
        *(" & ".join(r) + r" \\" for r in tex_rows),
        r"\bottomrule",
        r"\end{tabular}",
    ]
)

note = "nDCG calculado no split `test`, locale `us`, com ganhos ESCI: E=1.0, S=0.1, C=0.01, I=0.0."
(REPORT_DIR / "comparison_table.md").write_text(
    f"# Resultados do Benchmark\n\n{markdown_table}\n\n> {note}\n", encoding="utf-8"
)
(REPORT_DIR / "comparison_table.tex").write_text(latex_table + "\n", encoding="utf-8")
(
    df.with_columns(pl.col("model").replace(DISPLAY))
    .rename({"model": "modelo", "strategy": "estrategia"})
    .write_csv(REPORT_DIR / "comparison_table.csv", float_precision=4)
)
print(markdown_table)

## 3. Curva nDCG@k — melhor estratégia por modelo

In [ ]:
best_per_model = (
    df.drop_nulls("ndcg@10").sort("ndcg@10", descending=True).unique(subset="model", keep="first", maintain_order=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
for row in best_per_model.iter_rows(named=True):
    values = [row[f"ndcg@{k}"] for k in K_VALUES]
    if any(v is None for v in values):
        continue
    ax.plot(
        K_VALUES,
        values,
        label=f"{DISPLAY.get(row['model'], row['model'])} ({row['strategy']})",
        color=MODEL_COLORS.get(row["model"], "#333333"),
        linewidth=2,
        markersize=7,
        **STRATEGY_STYLES.get(row["strategy"], {"linestyle": "-", "marker": "x"}),
    )
ax.set_xlabel("k")
ax.set_ylabel("nDCG@k")
ax.set_title("Comparação de modelos — melhor estratégia por modelo")
ax.set_xticks(K_VALUES)
ax.set_ylim(bottom=0)
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "ndcg_curve.png", bbox_inches="tight")
plt.show()

## 4. Ranking por nDCG@10

In [ ]:
rank_df = df.drop_nulls("ndcg@10").sort("ndcg@10")
labels = [f"{DISPLAY.get(m, m)} / {s}" for m, s in zip(rank_df["model"], rank_df["strategy"])]
colors = [MODEL_COLORS.get(m, "#aaaaaa") for m in rank_df["model"]]

fig, ax = plt.subplots(figsize=(9, max(4, rank_df.height * 0.45)))
bars = ax.barh(labels, rank_df["ndcg@10"], color=colors, edgecolor="white", height=0.7)
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax.set_xlabel("nDCG@10")
ax.set_title("Ranking final por nDCG@10")
ax.set_xlim(0, min(1.0, rank_df["ndcg@10"].max() * 1.15))
ax.grid(True, axis="x", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "ndcg_bar.png", bbox_inches="tight")
plt.show()

## 5. Latência média por experimento

In [ ]:
lat_df = df.drop_nulls("latency_ms_mean").sort("latency_ms_mean", descending=True)
if lat_df.is_empty():
    print("Sem dados de latência.")
else:
    labels = [f"{DISPLAY.get(m, m)} / {s}" for m, s in zip(lat_df["model"], lat_df["strategy"])]
    colors = [MODEL_COLORS.get(m, "#aaaaaa") for m in lat_df["model"]]

    fig, ax = plt.subplots(figsize=(9, max(4, lat_df.height * 0.45)))
    bars = ax.barh(labels, lat_df["latency_ms_mean"], color=colors, edgecolor="white", height=0.7)
    ax.bar_label(bars, fmt="%.1f ms", padding=3, fontsize=8)
    ax.set_xlabel("Latência média por query (ms)")
    ax.set_title("Latência média de busca por modelo/estratégia")
    ax.grid(True, axis="x", linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "latency_bar.png", bbox_inches="tight")
    plt.show()

## 6. Hipóteses do TCC

Cada veredicto compara os valores agregados de nDCG@10. Os resultados por query não são salvos, portanto
**não há teste de significância estatística**: diferenças pequenas devem ser lidas com cautela.

In [ ]:
def best_ndcg10(model: str, strategies: tuple[str, ...] = ("knn", "hybrid")) -> float | None:
    values = df.filter((pl.col("model") == model) & pl.col("strategy").is_in(strategies))["ndcg@10"].drop_nulls()
    return values.max() if len(values) else None


def verdict(condition: bool | None) -> str:
    return "sem dados suficientes" if condition is None else ("CONFIRMADA" if condition else "NÃO CONFIRMADA")


def greater(a: float | None, b: float | None) -> bool | None:
    return None if a is None or b is None else a > b


def any_greater(candidates: list[float | None], reference: float | None) -> bool | None:
    present = [c for c in candidates if c is not None]
    return greater(max(present), reference) if present else None


v = {m: best_ndcg10(m) for m in MODEL_ORDER[1:]}


def show(name: str) -> str:
    return fmt(v.get(name))


print("H1: modelos especializados (BGE, RexBERT) superam o BERT genérico")
print(f"  BERT={show('bert')} | BGE={show('bge')} | RexBERT={show('rexbert')}")
print(f"  -> {verdict(any_greater([v.get('bge'), v.get('rexbert')], v.get('bert')))}\n")

print("H2: modelos de contexto longo (ModernBERT, RexBERT) superam o BERT de 512 tokens")
print(f"  BERT={show('bert')} | ModernBERT={show('modernbert')} | RexBERT={show('rexbert')}")
print(f"  -> {verdict(any_greater([v.get('modernbert'), v.get('rexbert')], v.get('bert')))}\n")

print("H3: BERTimbau tem desempenho inferior ao BERT em inglês")
print(f"  BERT={show('bert')} | BERTimbau={show('bertimbau')}")
print(f"  -> {verdict(greater(v.get('bert'), v.get('bertimbau')))}\n")

print("H4: Hybrid (RRF) supera o KNN puro na maioria dos modelos")
comparisons = [
    greater(best_ndcg10(m, ("hybrid",)), best_ndcg10(m, ("knn",)))
    for m in MODEL_ORDER[1:]
]
comparable = [c for c in comparisons if c is not None]
wins = sum(comparable)
print(f"  Modelos em que Hybrid > KNN: {wins}/{len(comparable)} comparáveis")
print(f"  -> {verdict(wins * 2 > len(comparable) if comparable else None)}")

## 7. Recomendações por cenário de uso

In [ ]:
rec_df = df.drop_nulls(["latency_ms_mean", "ndcg@10"]).with_columns(
    label=pl.col("model").replace(DISPLAY) + " / " + pl.col("strategy"),
    efficiency=pl.col("ndcg@10") / pl.col("latency_ms_mean"),
)
columns = ["label", "ndcg@10", "latency_ms_mean"]

if rec_df.is_empty():
    print("Dados de latência não disponíveis.")
else:
    print("Cenário 1 — máxima qualidade (latência não é restrição)")
    print(rec_df.sort("ndcg@10", descending=True).head(3).select(columns))
    print(f"\nCenário 2 — latência < {LATENCY_BUDGET_MS} ms e melhor qualidade")
    fast = rec_df.filter(pl.col("latency_ms_mean") < LATENCY_BUDGET_MS).sort("ndcg@10", descending=True).head(3)
    print(fast.select(columns) if not fast.is_empty() else f"Nenhum experimento abaixo de {LATENCY_BUDGET_MS} ms.")
    print("\nCenário 3 — melhor relação qualidade/latência (nDCG@10 por ms)")
    print(rec_df.sort("efficiency", descending=True).head(3).select([*columns, "efficiency"]))

## 8. Rascunho das conclusões

Texto gerado a partir dos números; revise e amplie com os achados do seu experimento.

In [ ]:
best = df.drop_nulls("ndcg@10").sort("ndcg@10", descending=True).row(0, named=True)
best_name = DISPLAY.get(best["model"], best["model"])

print(f"O melhor desempenho do benchmark foi de {best_name} com a estratégia {best['strategy'].upper()}, "
      f"com nDCG@10 = {best['ndcg@10']:.4f}.")

bm25_value = best_ndcg10("bm25", ("bm25",))
if bm25_value and best["model"] != "bm25":
    gain = (best["ndcg@10"] - bm25_value) / bm25_value * 100
    print(f"Isso representa {gain:+.1f}% em relação ao BM25 (nDCG@10 = {bm25_value:.4f}).")

## 9. Artefatos gerados

In [ ]:
for path in sorted(REPORT_DIR.iterdir()):
    print(f"{path.name:<40} {path.stat().st_size / 1024:8.1f} KB")